# 基于文档的QA

An example might be a tool that would allow you to query a product catalog for items of interest.

In [ ]:
#pip install --upgrade langchain

In [1]:
# 初始化
from langchain.chat_models import ChatOpenAI
import os
API_SECRET_KEY = "sk-scia8t0NThqBoFbDdzweXuPRmeQrMuQ9XkJaYl29VHyMACFD"
BASE_URL = "https://api.chatanywhere.tech"
os.environ["OPENAI_API_KEY"] = API_SECRET_KEY
os.environ["OPENAI_API_BASE"] = BASE_URL
llm_model = "gpt-3.5-turbo"

In [2]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader
# 导入文档向量存储
from langchain.vectorstores import DocArrayInMemorySearch
from IPython.display import display, Markdown
from langchain.llms import OpenAI

# 户外服装目录：名字，描述
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)

# 导入向量存储索引器
from langchain.indexes import VectorstoreIndexCreator
#pip install docarray
# 创建向量存储实例，传入向量存储类型和n个加载器
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch
).from_loaders([loader])

# 创建好了向量存储，可以开始基于文档提问了
query ="Please list all your shirts with sun protection \
in a table in markdown and summarize each one."

**Note**:
- The notebook uses `langchain==0.0.179` and `openai==0.27.7`
- For these library versions, `VectorstoreIndexCreator` uses `text-davinci-003` as the base model, which has been deprecated since 1 January 2024.
- The replacement model, `gpt-3.5-turbo-instruct` will be used instead for the `query`.
- The `response` format might be different than the video because of this replacement model.

### 下面就可以基于文档进行问答了，为什么能做到这点，LLM一次只能输入那么少的词，不可能把整个文档传过去

1. 首先将整个文档向量化，即embedding，每个词都映射到一个多维度的向量
2. 比较所有文档所有片段和片段之间的相似度
3. 找出和问题相似的片段，传递给llm

### 向量数据库

我们在创建索引实例的时候发生的事情！！

1. 将文档拆分成若干块，每块生成对应的embedding
2. 将embedding和对应原始文字块存储到数据库中
3. 找到和查询内容相关的几个块，将对应的文本块去除和查询内容传递给llm

In [ ]:
llm_replacement_model = ChatOpenAI(temperature=0, 
                               model='gpt-3.5-turbo')

response = index.query(query, llm = llm_replacement_model)

In [5]:
display(Markdown(response))

| Shirt Name | Description |
|------------|-------------|
| Men's Tropical Plaid Short-Sleeve Shirt | Rated UPF 50+ for superior sun protection, made of 100% polyester, wrinkle-resistant, with front and back cape venting, and two front bellows pockets. Provides the highest rated sun protection possible. |
| Men's Plaid Tropic Shirt, Short-Sleeve | UPF 50+ sun protection, designed for fishing and extended travel, blocks 98% of harmful UV rays, made of 52% polyester and 48% nylon, wrinkle-free, with front and back cape venting, two front bellows pockets. |
| Sun Shield Shirt | High-performance sun shirt with UPF 50+ sun protection, made of 78% nylon and 22% Lycra Xtra Life fiber, wicks moisture, fits comfortably over swimsuits, abrasion-resistant. Recommended by The Skin Cancer Foundation. |
| Men's TropicVibe Shirt, Short-Sleeve | Men's sun-protection shirt with built-in UPF 50+, lightweight feel, traditional fit, made of 71% Nylon and 29% Polyester, wrinkle-resistant, front and back cape venting, two front bellows pockets. Provides SPF 50+ sun protection. |

## 深入了解底层是怎么做的

In [6]:
from langchain.document_loaders import CSVLoader
loader = CSVLoader(file_path=file)

In [7]:
docs = loader.load()

In [ ]:
# 查看第一个产品
docs[0]

Document(page_content=": 2\nname: Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece\ndescription: She'll love the bright colors, ruffles and exclusive whimsical prints of this toddler's two-piece swimsuit! Our four-way-stretch and chlorine-resistant fabric keeps its shape and resists snags. The UPF 50+ rated fabric provides the highest rated sun protection possible, blocking 98% of the sun's harmful rays. The crossover no-slip straps and fully lined bottom ensure a secure fit and maximum coverage. Machine wash and line dry for best results. Imported.", metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 2})

In [ ]:
# 使用openAi的Embedding方式
from langchain.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
# 手动对一句话进行embedding
embed = embeddings.embed_query("Hi my name is Harrison")
print(len(embed))
# 这里和深度学习的embedding不太一样，深度学习是对单词。或者字母作为词元，对每个词元进行token，这里是将整个句子作为词元tokenize
print(embed[:5])

1536
[-0.021964654326438904, 0.006758837960660458, -0.01824948936700821, -0.03923514857888222, -0.014007173478603363]


In [18]:
# 得到向量数据库
# 对文档所有行（产品）生成对应embedding
db = DocArrayInMemorySearch.from_documents(
    docs, 
    embeddings
)

In [ ]:
query = "Please suggest a shirt with sunblocking"
docs = db.similarity_search(query)  # 将问题输入到数据库中，自动embedding问题并找出相似的几个文本块
print(len(docs))
print(docs[0])

4
page_content=': 255\nname: Sun Shield Shirt by\ndescription: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. \r\n\r\nSize & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.\r\n\r\nFabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the highest rated sun protection possible. Handwash, line dry.\r\n\r\nAdditional Features: Wicks moisture for quick-drying comfort. Fits comfortably over your favorite swimsuit. Abrasion resistant for season after season of wear. Imported.\r\n\r\nSun Protection That Won\'t Wear Off\r\nOur high-performance fabric provides SPF 50+ sun protection, blocking 98% of the sun\'s harmful rays. This fabric is recommended by The Skin Cancer Foundation as an effective UV protectant.' metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 255}


In [ ]:
retriever = db.as_retriever() # 创建检索器

In [22]:
# 手动将刚刚的问题相似片段和问题输入llm得到答案
llm = ChatOpenAI(temperature = 0.0, model=llm_model)
qdocs = "".join([docs[i].page_content for i in range(len(docs))])

response = llm.call_as_llm(f"{qdocs} Question: Please list all your \
shirts with sun protection in a table in markdown and summarize each one.") 
# markdown格式展示
display(Markdown(response))

| Name                           | Description                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [24]:
# langchain的快捷接口
# 创建QA链，这个链对查询进行检索，然后将检索结果和问题组合输入llm
qa_stuff = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=retriever, 
    verbose=True
)

In [ ]:
query =  "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

In [ ]:
response = qa_stuff.run(query)

In [ ]:
display(Markdown(response))

In [ ]:
response = index.query(query, llm=llm)

In [ ]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings,
).from_loaders([loader])